# KAN-MCP 最小复现（CMU-MOSI / CMU-MOSEI）

按顺序运行。使用 Colab GPU 运行环境；日志、指标和运行配置写入 Google Drive 的 `MyDrive/KAN-MCP-reproduction/<run_id>/`。论文报告五次实验均值；默认每个数据集只运行一次，确认流程后可把 `SEEDS` 改为五个种子。当前公开仓库没有 CH-SIMS v2 训练入口。

论文：[KAN-MCP.pdf](https://drive.google.com/file/d/1qPrmCZlPIvZ-cQ0NPVu1voXqFkgmnWxL/view)；代码：[LuoMSen/KAN-MCP](https://github.com/LuoMSen/KAN-MCP)。

In [ ]:
from google.colab import drive
import subprocess

drive.mount('/content/drive')
subprocess.run(['nvidia-smi'], check=True)

# 修改这些配置后，从此格开始按顺序重新运行。
REPO_URL = 'https://github.com/Scream9371/KAN-MCP.git'
REPO_REF = 'main'  # 每次读取 Fork 最新 main；实际 commit 会写入 manifest.json
DATASETS = ['mosi', 'mosei']  # 可只保留一个，先做较小的 MOSI
SEEDS = {'mosi': [1024], 'mosei': [1025]}
DRIVE_ROOT = '/content/drive/MyDrive/KAN-MCP-reproduction'

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, sys

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_dir = Path(DRIVE_ROOT) / run_id
out_dir.mkdir(parents=True, exist_ok=False)
repo_dir = Path('/content') / f'KAN-MCP-{run_id}'
if not repo_dir.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', commit)
print('Drive output:', out_dir)

In [ ]:
# 依赖锁定了 numpy 1.22.4；用 Python 3.10 隔离环境，避免与 Colab 当前内核版本冲突。
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.10'], check=True)
venv_dir = Path('/content/kanmcp-venv')
subprocess.run(['uv', 'venv', '--python', '3.10', str(venv_dir)], check=True)
train_python = str(venv_dir / 'bin' / 'python')
subprocess.run(['uv', 'pip', 'install', '--python', train_python, '-r', str(repo_dir / 'requirements.txt')], check=True)
probe = subprocess.run([train_python, '-c', 'import sys,torch,transformers; print(sys.version.split()[0], torch.__version__, torch.version.cuda, transformers.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'], capture_output=True, text=True)
print(probe.stdout, probe.stderr)
probe.check_returncode()


## 数据

优先使用仓库 `datasets/` 中现有文件。若从 GitHub 克隆的仓库不含数据，按 README 给出的链接下载。MOSEI 文件较大，下载和磁盘空间需预留时间。若数据已在 Drive，可以把文件手动放入克隆仓库的 `datasets/` 后运行下一格。

In [ ]:
import urllib.request

urls = {
    'mosi': 'https://www.dropbox.com/s/sv94igp7zi3rsj1/mosi.pkl?dl=1',
    'mosei': 'https://www.dropbox.com/s/995dvj3506gvk5a/mosei.pkl?dl=1',
}
for name in DATASETS:
    target = repo_dir / 'datasets' / f'{name}.pkl'
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_suffix('.pkl.part')
        urllib.request.urlretrieve(urls[name], temp)
        temp.replace(target)
    assert target.stat().st_size > 1_000_000, f'数据文件异常：{target}'
    print(name, f'{target.stat().st_size / 1024**2:.1f} MiB')

## 训练

当前 Fork 的训练脚本应由仓库直接维护，不在 Colab 中动态修改。每个数据集和种子独立运行；标准输出同时写入 Drive。脚本最后的 `====RESULT====` 是验证损失最低 epoch 对应的测试指标。

In [ ]:
import re, csv, time

for name in DATASETS:
    script = (repo_dir / f'train_{name}.py').read_text()
    assert 'CUDA_VISIBLE_DEVICES' not in script, f'{name} 脚本仍硬编码 GPU；请从已更新的 Fork 拉取。'

versions = subprocess.check_output([train_python, '-c', 'import sys,torch,transformers; print(sys.version.split()[0],torch.__version__,torch.version.cuda,transformers.__version__,torch.cuda.get_device_name(0))'], text=True).strip().split(' ', 4)
manifest = {
    'repo_url': REPO_URL, 'commit': commit, 'datasets': DATASETS, 'seeds': SEEDS,
    'python': versions[0], 'torch': versions[1], 'cuda': versions[2],
    'transformers': versions[3], 'gpu': versions[4],
    'note': 'Training scripts are used directly from the Fork without runtime edits.'
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print('配置已保存:', out_dir / 'manifest.json')


In [ ]:
metric_pattern = re.compile(r'acc7:([^,]+), acc2:([^,]+), f1:([^,]+), mae:([^,]+), corr:([^\s]+)')
rows = []
for name in DATASETS:
    for seed in SEEDS[name]:
        log_path = out_dir / f'{name}_seed{seed}.log'
        command = [train_python, f'train_{name}.py', '--seed', str(seed)]
        print('Running:', ' '.join(command), '| log:', log_path, flush=True)
        start = time.time()
        with log_path.open('w', encoding='utf-8') as log:
            process = subprocess.Popen(command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
            tail = []
            for line in process.stdout:
                log.write(line)
                log.flush()
                tail.append(line.strip())
                tail = tail[-12:]
                if 'TRAIN:' in line or 'TEST:' in line or '====RESULT====' in line or 'acc7:' in line:
                    print(line.rstrip(), flush=True)
            rc = process.wait()
        if rc != 0:
            raise RuntimeError(f'{name} seed={seed} 失败（退出码 {rc}），查看 {log_path}；末尾输出：{tail}')
        log_text = log_path.read_text(encoding='utf-8')
        result_block = log_text.rsplit('====RESULT====', 1)
        assert len(result_block) == 2, f'缺少最终结果：{log_path}'
        match = metric_pattern.search(result_block[1])
        assert match, f'无法解析最终结果：{log_path}'
        row = {'dataset': name, 'seed': seed, 'seconds': round(time.time()-start, 1),
               **dict(zip(['acc7','acc2','f1','mae','corr'], map(float, match.groups())))}
        rows.append(row)
        (out_dir / 'results.json').write_text(json.dumps(rows, indent=2, ensure_ascii=False))
        print('Saved:', row, flush=True)
print('全部完成，结果目录:', out_dir)

In [ ]:
import statistics

rows = json.loads((out_dir / 'results.json').read_text())
fields = ['acc7', 'acc2', 'f1', 'mae', 'corr']
summary = []
for name in DATASETS:
    selected = [r for r in rows if r['dataset'] == name]
    if not selected:
        continue
    item = {'dataset': name, 'runs': len(selected)}
    item.update({field: statistics.mean(r[field] for r in selected) for field in fields})
    summary.append(item)
(out_dir / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
for item in summary:
    print(item)
print('论文表 1：MOSI Acc7 48.3%, Acc2 89.4%, F1 89.4%, MAE 0.615, Corr 0.857；MOSEI 53.9%, 87.7%, 87.6%, 0.522, 0.788。')
print('注意：上面 Acc/F1 为 0–1 比例；论文以百分数呈现。')
print('结果已持久化于:', out_dir)